# Match Info Processor — KDE replacement

这是独立的新版本，不修改原始 `Match_Info_Processor.ipynb`。原有比赛信息字段保持不变；`Team_Heatmap(Home)`、`Team_Heatmap(Away)` 和 `Player_Heatmap` 改为直接由 event 坐标生成的二维 Gaussian KDE 强度场。

数值结果使用 `(height, width)` 排列、`float32` 保存；数组第 0 行对应 `y=0`。绘图时使用 `origin='lower'`。

In [1]:
import os
import time
import warnings
from pathlib import Path

import orjson
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm.auto import tqdm


## Configuration

`BUILD_PLAYER_KDE=True` 与原 notebook 字段完全对应，但会明显增加内存和输出文件体积；只需要球队 KDE 时可设为 `False`。

In [2]:
DATA_ROOT = Path(r'D:\code\project (Optimal Transport)\Data Extractor and Processor')
OUTPUT_FILE = DATA_ROOT / 'dataset_kde.pkl'  # 不覆盖原来的 dataset.pkl

LEAGUES = [
    'England_Premier_League',
    'Spain_Laliga',
    'Italy_Serie_A',
    'Germany_Bundesliga',
    'France_Ligue1',
]

IS_TOUCH = True
ROTATE_AWAY_180 = True

KDE_WIDTH = 112
KDE_HEIGHT = 72
BW_ADJUST = 0.50
GRID_CHUNK_SIZE = 1024
BUILD_PLAYER_KDE = True
STORE_DTYPE = np.float16

N_JOBS = min(4, os.cpu_count() or 1)
RUN_BATCH = True # 完成单场验证后，再改为 True 生成完整 dataset_kde.pkl

KDE_CONFIG = {
    'width': KDE_WIDTH,
    'height': KDE_HEIGHT,
    'bw_adjust': BW_ADJUST,
    'normalization': 'event_intensity',
    'away_rotation_180': ROTATE_AWAY_180,
    'array_layout': '[y, x], row 0 is y=0',
    'dtype': np.dtype(STORE_DTYPE).name,
}

## General helpers

In [3]:
def safe_parse_score(score):
    try:
        home_score, away_score = score.replace(' ', '').split(':')
        home_score = int(home_score)
        away_score = int(away_score)
        return home_score, away_score, home_score - away_score
    except Exception:
        return None, None, None


def is_starting_player(player):
    if 'isFirstEleven' in player:
        return bool(player.get('isFirstEleven'))
    try:
        return '0' in player.get('stats', {}).get('ratings', {})
    except Exception:
        return False


def build_player_attribute_dict(players, attr_name):
    return {
        player['playerId']: player.get(attr_name)
        for player in players
        if player.get('playerId') is not None
    }


def build_player_rating_dict(players):
    return {
        player['playerId']: next(
            reversed(player['stats'].get('ratings', {}).values()), None
        )
        for player in players
        if player.get('stats') is not None and player.get('playerId') is not None
    }


def unique_player_ids(players):
    # 保持 JSON 中的稳定顺序。
    return list(dict.fromkeys(
        player['playerId']
        for player in players
        if player.get('playerId') is not None
    ))

## Optimized KDE engine

每支球队只估计一次 Scott 带宽矩阵。球队和该队所有球员共用这个核；随后按网格块一次性计算事件核，并用事件—球员成员矩阵同时聚合球队及所有球员 KDE。这样避免为每名球员重复拟合和重复扫描全部网格，也使零触球/单触球球员保持稳定。

In [4]:
def get_evaluation_grid(width, height):
    x = np.linspace(0.0, 100.0, width, dtype=np.float64)
    y = np.linspace(0.0, 100.0, height, dtype=np.float64)
    grid_x, grid_y = np.meshgrid(x, y)
    return np.column_stack([grid_x.ravel(), grid_y.ravel()])


def estimate_bandwidth_matrix(points, bw_adjust):
    """Scott bandwidth used by a two-dimensional Gaussian KDE."""
    n_events = len(points)
    if n_events >= 2:
        sample_covariance = np.asarray(np.cov(points, rowvar=False), dtype=np.float64)
        if sample_covariance.shape != (2, 2) or not np.isfinite(sample_covariance).all():
            sample_covariance = np.eye(2, dtype=np.float64) * 100.0
        scott_factor = n_events ** (-1.0 / 6.0)
        bandwidth = sample_covariance * (scott_factor * bw_adjust) ** 2
    else:
        # 仅用于极端的空/单事件输入；正常球队不会走到这里。
        bandwidth = np.eye(2, dtype=np.float64) * 25.0

    # 只在协方差退化时加入极小正则项。
    eigenvalues, eigenvectors = np.linalg.eigh(bandwidth)
    maximum = max(float(np.max(eigenvalues)), 1.0)
    floor = maximum * 1e-6
    eigenvalues = np.maximum(eigenvalues, floor)
    return (eigenvectors * eigenvalues) @ eigenvectors.T


def build_team_and_player_kde(
    coordinates,
    event_player_ids,
    roster_player_ids,
    width=364,
    height=238,
    bw_adjust=0.50,
    chunk_size=1024,
    build_player_kde=True,
):
    """Return event-intensity KDEs for one team and all of its players."""
    grid = get_evaluation_grid(width, height)
    grid_size = len(grid)
    output_shape = (height, width)

    roster_player_ids = list(roster_player_ids)
    zero_map = np.zeros(output_shape, dtype=STORE_DTYPE)
    if len(coordinates) == 0:
        player_maps = (
            {player_id: zero_map.copy() for player_id in roster_player_ids}
            if build_player_kde else {}
        )
        return zero_map, player_maps

    points = np.asarray(coordinates, dtype=np.float64).reshape(-1, 2)
    event_player_ids = np.asarray(event_player_ids)
    bandwidth = estimate_bandwidth_matrix(points, bw_adjust)
    inverse_bandwidth = np.linalg.inv(bandwidth)
    determinant = float(np.linalg.det(bandwidth))
    normalizer = 1.0 / (2.0 * np.pi * np.sqrt(determinant))

    team_flat = np.empty(grid_size, dtype=STORE_DTYPE)

    if build_player_kde:
        player_to_column = {
            player_id: column for column, player_id in enumerate(roster_player_ids)
        }
        membership = np.zeros(
            (len(points), len(roster_player_ids)), dtype=STORE_DTYPE
        )
        for event_index, player_id in enumerate(event_player_ids):
            column = player_to_column.get(player_id)
            if column is not None:
                membership[event_index, column] = 1.0
        player_flat = np.empty(
            (len(roster_player_ids), grid_size), dtype=STORE_DTYPE
        )

    for start in range(0, grid_size, chunk_size):
        stop = min(start + chunk_size, grid_size)
        delta = grid[start:stop, None, :] - points[None, :, :]
        mahalanobis = np.einsum(
            'cni,ij,cnj->cn',
            delta, inverse_bandwidth, delta,
            optimize=True,
        )
        np.maximum(mahalanobis, 0.0, out=mahalanobis)
        kernels = np.exp(-0.5 * mahalanobis)
        kernels *= normalizer
        kernels = kernels.astype(STORE_DTYPE, copy=False)

        # Sum of event kernels: N * scipy gaussian_kde probability density.
        team_flat[start:stop] = kernels.sum(axis=1, dtype=STORE_DTYPE)
        if build_player_kde:
            player_flat[:, start:stop] = (kernels @ membership).T

    team_map = team_flat.reshape(output_shape)
    if build_player_kde:
        player_maps = {
            player_id: player_flat[column].reshape(output_shape)
            for column, player_id in enumerate(roster_player_ids)
        }
    else:
        player_maps = {}

    return team_map, player_maps

## Match processor

In [5]:
def process_json(league_name, json_path):
    with open(json_path, 'rb') as file:
        data = orjson.loads(file.read())

    home_players = data['home']['players']
    away_players = data['away']['players']
    players = home_players + away_players

    player_age_dict = build_player_attribute_dict(players, 'age')
    player_weight_dict = build_player_attribute_dict(players, 'weight')
    player_height_dict = build_player_attribute_dict(players, 'height')
    player_position_dict = build_player_attribute_dict(players, 'position')
    player_rating_dict = build_player_rating_dict(players)

    starting_11_home = [
        player['playerId'] for player in home_players
        if player.get('playerId') is not None and is_starting_player(player)
    ]
    substitute_home = [
        player['playerId'] for player in home_players
        if player.get('playerId') is not None and not is_starting_player(player)
    ]
    starting_11_away = [
        player['playerId'] for player in away_players
        if player.get('playerId') is not None and is_starting_player(player)
    ]
    substitute_away = [
        player['playerId'] for player in away_players
        if player.get('playerId') is not None and not is_starting_player(player)
    ]

    if len(starting_11_home) != 11:
        warnings.warn(f'{json_path}: home starting players = {len(starting_11_home)}')
    if len(starting_11_away) != 11:
        warnings.warn(f'{json_path}: away starting players = {len(starting_11_away)}')

    home_roster = unique_player_ids(home_players)
    away_roster = unique_player_ids(away_players)
    home_player_set = set(home_roster)
    away_player_set = set(away_roster)

    home_coordinates = []
    away_coordinates = []
    home_event_players = []
    away_event_players = []

    for event in data['events']:
        if IS_TOUCH and not event.get('isTouch', False):
            continue

        player_id = event.get('playerId')
        x = event.get('x')
        y = event.get('y')
        if player_id is None or x is None or y is None:
            continue

        x = float(x)
        y = float(y)
        if not (0.0 <= x <= 100.0 and 0.0 <= y <= 100.0):
            continue

        if player_id in home_player_set:
            home_coordinates.append((x, y))
            home_event_players.append(player_id)
        elif player_id in away_player_set:
            if ROTATE_AWAY_180:
                x, y = 100.0 - x, 100.0 - y
            away_coordinates.append((x, y))
            away_event_players.append(player_id)

    home_kde, home_player_kde = build_team_and_player_kde(
        home_coordinates,
        home_event_players,
        home_roster,
        width=KDE_WIDTH,
        height=KDE_HEIGHT,
        bw_adjust=BW_ADJUST,
        chunk_size=GRID_CHUNK_SIZE,
        build_player_kde=BUILD_PLAYER_KDE,
    )
    away_kde, away_player_kde = build_team_and_player_kde(
        away_coordinates,
        away_event_players,
        away_roster,
        width=KDE_WIDTH,
        height=KDE_HEIGHT,
        bw_adjust=BW_ADJUST,
        chunk_size=GRID_CHUNK_SIZE,
        build_player_kde=BUILD_PLAYER_KDE,
    )
    player_kde = {**home_player_kde, **away_player_kde}

    home_score, away_score, score_diff = safe_parse_score(data.get('score'))

    return {
        'League': league_name,
        'Match_Date': data['startTime'],
        'Home_Team': data['home']['name'],
        'Away_Team': data['away']['name'],
        'Home_Team_ID': data['home']['teamId'],
        'Away_Team_ID': data['away']['teamId'],
        'Home_Score': home_score,
        'Away_Score': away_score,
        'Score_Diff': score_diff,
        'Starting_11(Home)': starting_11_home,
        'Substitute(Home)': substitute_home,
        'Starting_11(Away)': starting_11_away,
        'Substitute(Away)': substitute_away,
        'Manager(Home)': data['home']['managerName'],
        'Manager(Away)': data['away']['managerName'],
        'Referee': data['referee']['officialId'],
        'Player_Age': player_age_dict,
        'Player_Weight': player_weight_dict,
        'Player_Height': player_height_dict,
        'Player_Position': player_position_dict,
        'Player_Rating': player_rating_dict,
        'Team_Heatmap(Home)': home_kde,
        'Team_Heatmap(Away)': away_kde,
        'Player_Heatmap': player_kde,
        'Touch_Count(Home)': len(home_coordinates),
        'Touch_Count(Away)': len(away_coordinates),
    }

## Batch processing

把 `RUN_BATCH=True` 后执行此单元。结果写入新的 `dataset_kde.pkl`，不会覆盖原 `dataset.pkl`。

In [6]:
if RUN_BATCH:
    tasks = []
    for league_name in LEAGUES:
        league_directory = DATA_ROOT / league_name
        if not league_directory.is_dir():
            warnings.warn(f'Skipping missing directory: {league_directory}')
            continue
        tasks.extend(
            (league_name, json_path)
            for json_path in league_directory.glob('*.json')
        )

    print(f'Matches to process: {len(tasks)}')
    batch_start = time.perf_counter()
    results = Parallel(
        n_jobs=N_JOBS,
        backend='loky',
        batch_size=1,
        verbose=10,
    )(
        delayed(process_json)(league_name, json_path)
        for league_name, json_path in tasks
    )

    match_df = pd.DataFrame(results)
    match_df.attrs['KDE_CONFIG'] = KDE_CONFIG
    match_df.to_pickle(OUTPUT_FILE)
    print(f'Saved: {OUTPUT_FILE}')
    print(f'Elapsed: {time.perf_counter() - batch_start:.2f} s')
else:
    print('RUN_BATCH=False: full dataset was not generated.')

Matches to process: 19714


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    2.3s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    4.0s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    8.6s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:   11.2s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   17.4s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:   21.8s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:   28.2s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:   33.7s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:   41.5s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:   47.9s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:   56.2s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:  1.1min
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:  1.2min
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:  1.4min
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:  1.6min
[Parallel(

[Parallel(n_jobs=4)]: Done 9377 tasks      | elapsed: 95.0min
[Parallel(n_jobs=4)]: Done 9514 tasks      | elapsed: 96.4min
[Parallel(n_jobs=4)]: Done 9653 tasks      | elapsed: 97.9min
[Parallel(n_jobs=4)]: Done 9792 tasks      | elapsed: 99.4min
[Parallel(n_jobs=4)]: Done 9933 tasks      | elapsed: 100.9min
[Parallel(n_jobs=4)]: Done 10074 tasks      | elapsed: 102.4min
[Parallel(n_jobs=4)]: Done 10217 tasks      | elapsed: 104.0min
[Parallel(n_jobs=4)]: Done 10360 tasks      | elapsed: 105.5min
[Parallel(n_jobs=4)]: Done 10505 tasks      | elapsed: 107.1min
[Parallel(n_jobs=4)]: Done 10650 tasks      | elapsed: 108.6min
[Parallel(n_jobs=4)]: Done 10797 tasks      | elapsed: 110.2min
[Parallel(n_jobs=4)]: Done 10944 tasks      | elapsed: 111.8min
[Parallel(n_jobs=4)]: Done 11093 tasks      | elapsed: 113.4min
[Parallel(n_jobs=4)]: Done 11242 tasks      | elapsed: 115.0min
[Parallel(n_jobs=4)]: Done 11393 tasks      | elapsed: 116.7min
[Parallel(n_jobs=4)]: Done 11544 tasks      | ela

Saved: D:\code\project (Optimal Transport)\Data Extractor and Processor\dataset_kde.pkl
Elapsed: 12290.83 s
